# REMNET Stage 3 — D1 vs D3 Classification Paired Statistical Analysis
**Direct numerical input edition · adapted from `classification-test.ipynb` using measured `part3.zip` D3 outputs.**

No dataset paths, images, checkpoints, or internet required. D1 uses the five-seed numbers retained from the user-supplied reference notebook (**463 test images**). D3 uses the five-seed, full-precision classification CSV values from `part3.zip` (**3,497 independent external images**). Seeds: **7, 49, 112, 345, 476**.

Descriptive results are mean ± sample SD. The six primary paired comparisons are REMNET versus three baselines on **D3 accuracy and macro-F1**; Holm correction is applied over this six-test family. Precision and recall are exploratory, with a separate six-test Holm adjustment. Paired Student-t intervals (df=4), Cohen dz, and exact sign-flip sensitivity p-values are reported. **Matching seed numbers is not proof of independence or statistical superiority.** In particular, five paired seeds give a minimum two-sided exact sign-flip p-value of 0.0625.

D1 and D3 may differ in collection conditions and class distributions; the D3−D1 difference measures performance change across datasets, not a causal geographic effect. No D3 labels or predictions are used to modify model parameters.


In [1]:
# Stage 3 statistical analysis; all five-seed inputs are literal numbers below.
# No paths, file loading, checkpoint loading, GPUs or internet are necessary.
import itertools
import numpy as np
import pandas as pd
from scipy import stats

SEEDS = [7, 49, 112, 345, 476]
MODELS = ['REMNET', 'ResNet50', 'MobileNetV2', 'EfficientNet-Lite0']
BASELINES = ['ResNet50', 'MobileNetV2', 'EfficientNet-Lite0']
ALPHA = 0.05
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 230)
pd.set_option('display.max_columns', 32)
pd.set_option('display.float_format', lambda x: f'{x:.12f}')

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    assert np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all()
    ranks = np.argsort(p, kind='stable')
    out = np.empty(len(p), dtype=float)
    current = 0.0
    for rank, idx in enumerate(ranks):
        current = max(current, min(1.0, (len(p)-rank)*p[idx]))
        out[idx] = current
    return out

def paired_test(remnet, baseline):
    a, b = np.asarray(remnet, dtype=float), np.asarray(baseline, dtype=float)
    assert len(a) == len(b) == 5 and np.isfinite(a).all() and np.isfinite(b).all()
    d = a-b  # REMNET minus CNN; negative energy difference means lower recorded energy
    mean, sd = float(d.mean()), float(d.std(ddof=1))
    if sd == 0:
        raise ValueError('Paired differences have zero sample SD; t-statistic / dz undefined')
    t = mean/(sd/np.sqrt(5))
    p = float(2*stats.t.sf(abs(t), df=4))
    half_width = float(stats.t.ppf(0.975,df=4) * sd/np.sqrt(5))
    # Exact sign-flip randomization sensitivity test: only 32 possible flips at n=5.
    flips = [np.mean(d*np.array(s)) for s in itertools.product((-1,1),repeat=5)]
    exact_p = float(np.mean(np.abs(flips) >= abs(mean)-1e-14))
    return dict(mean_difference=mean, sample_SD_difference=sd,
      CI95_lower=mean-half_width, CI95_upper=mean+half_width,
      Cohen_dz=mean/sd, paired_t=t, df=4,
      p_two_sided=p, exact_sign_flip_p_two_sided=exact_p)

def report(df, cols=None, digits=9):
    print((df if cols is None else df[cols]).to_string(index=False,
       float_format=lambda x:f'{x:.{digits}f}'))


## 1. Original D1 measurements and current D3 five-seed classification results (embedded values)


In [2]:
# Metric values are percentages. All five values are ordered by SEEDS.
# D1: retained exactly from uploaded classification-test.ipynb.
# D3: extracted from model-specific *All_Seeds_D3_External.csv files in part3.zip.
classification_values = {'D1': {'REMNET': {'accuracy': [94.6004319654, 94.8164146868, 93.7365010799, 94.1684665227, 93.3045356371],
                   'macro_precision': [94.6181563724, 94.8338353105, 93.5746492541, 94.5726494876, 93.507016715],
                   'macro_recall': [93.3759125241, 93.4488365942, 91.9698058148, 92.4308189454, 91.4288665475],
                   'macro_f1': [93.7913961276, 93.9636710546, 92.4940942008, 93.1294168986, 92.0023203797]},
        'ResNet50': {'accuracy': [90.4967602592, 92.0086393089, 93.9524838013, 90.0647948164, 91.5766738661],
                     'macro_precision': [90.3510351129, 92.2721493901, 93.5497363453, 89.8398123403, 91.677918006],
                     'macro_recall': [89.9789207852, 90.6339778089, 93.3886041679, 88.7591888266, 89.208093288],
                     'macro_f1': [89.8109998638, 91.1763004168, 93.4578801144, 89.0000212123, 89.5541828505]},
        'MobileNetV2': {'accuracy': [90.9287257019, 90.9287257019, 90.0647948164, 89.848812095, 91.1447084233],
                        'macro_precision': [91.900203513,
                                            90.781480856,
                                            91.7567835925,
                                            90.9072147106,
                                            92.2053810994],
                        'macro_recall': [88.6450697959,
                                         89.7542245334,
                                         87.5865975324,
                                         87.7150583754,
                                         89.0262421653],
                        'macro_f1': [89.3617203452, 90.1644315014, 88.4871125146, 88.6001370824, 89.8809676401]},
        'EfficientNet-Lite0': {'accuracy': [90.2807775378,
                                            87.4730021598,
                                            90.7127429806,
                                            90.4967602592,
                                            90.4967602592],
                               'macro_precision': [90.5138290485,
                                                   87.5412244858,
                                                   90.8765504493,
                                                   90.9418445927,
                                                   91.0791525246],
                               'macro_recall': [88.7259898565,
                                                85.2303613632,
                                                88.4714586848,
                                                88.0354166182,
                                                88.6662429238],
                               'macro_f1': [89.3825996851,
                                            85.6629376817,
                                            89.0031803678,
                                            88.6153323416,
                                            89.3373558827]}},
 'D3': {'REMNET': {'accuracy': [46.0966542751, 39.5767800972, 47.2118959108, 42.9511009437, 48.0125822133],
                   'macro_precision': [53.0738536662, 47.533047447, 50.5105000254, 56.3343669712, 56.686814145],
                   'macro_recall': [43.6272161457, 37.7603770922, 45.1487672824, 40.9067767918, 46.3597347185],
                   'macro_f1': [41.1166336769, 35.3472645484, 43.612793315, 37.912279274, 45.2520656781]},
        'ResNet50': {'accuracy': [34.9728338576, 40.6348298542, 43.58021161, 37.8610237346, 43.7517872462],
                     'macro_precision': [37.4564564167,
                                         42.7591039221,
                                         42.6099621397,
                                         43.5327251813,
                                         43.5723580957],
                     'macro_recall': [33.4322937997, 40.8255082358, 39.2390802958, 36.4575039088, 41.6642473062],
                     'macro_f1': [31.1152675096, 39.4105929372, 39.8624969887, 35.1608326781, 39.2518906073]},
        'MobileNetV2': {'accuracy': [37.5464684015, 33.3142693737, 37.6894480984, 41.0637689448, 31.2553617386],
                        'macro_precision': [42.376264056,
                                            46.0030793066,
                                            42.5117930941,
                                            48.9319926957,
                                            38.9589855073],
                        'macro_recall': [39.2910006958,
                                         35.8173958556,
                                         37.4784033145,
                                         41.1469430425,
                                         33.1858942423],
                        'macro_f1': [37.2119255622, 31.0148975868, 34.2443618108, 39.5427581749, 28.5663947002]},
        'EfficientNet-Lite0': {'accuracy': [41.1781527023,
                                            33.1140977981,
                                            38.4615384615,
                                            34.3723191307,
                                            37.6322562196],
                               'macro_precision': [44.3127347082,
                                                   37.6613906029,
                                                   41.4124792624,
                                                   42.0676150073,
                                                   41.251129595],
                               'macro_recall': [40.5086272293,
                                                33.8822324943,
                                                36.8571220979,
                                                34.7924919819,
                                                38.9659454343],
                               'macro_f1': [37.5003128868,
                                            32.713227238,
                                            33.8063538379,
                                            31.3817888517,
                                            36.2301767936]}}}


## 2. Raw five-seed data, mean ± sample SD, D3 − D1 change


In [3]:
METRICS = ['accuracy','macro_precision','macro_recall','macro_f1']
raw_rows=[]
for dataset in ['D1','D3']:
    for model in MODELS:
        for i, seed in enumerate(SEEDS):
            raw_rows.append({'dataset':dataset,'model':model,'seed':seed,
                             'n_images':463 if dataset=='D1' else 3497,
                             **{metric: classification_values[dataset][model][metric][i] for metric in METRICS}})
raw_classification = pd.DataFrame(raw_rows)
assert len(raw_classification)==40
summary_rows=[]
for dataset in ['D1','D3']:
    for model in MODELS:
        for metric in METRICS:
            v=np.asarray(classification_values[dataset][model][metric],dtype=float)
            summary_rows.append(dict(dataset=dataset,model=model,metric=metric,
                        mean_pct=float(v.mean()),sample_SD_pct=float(v.std(ddof=1))))
classification_summary = pd.DataFrame(summary_rows)
print('FIVE-SEED CLASSIFICATION MEAN AND SAMPLE SD (percentage units):')
report(classification_summary, digits=6)

change_rows=[]
for model in MODELS:
    for i,seed in enumerate(SEEDS):
        change_rows.append({'model':model,'seed':seed,**{
            metric+'_D3_minus_D1_pp':classification_values['D3'][model][metric][i]-classification_values['D1'][model][metric][i]
            for metric in METRICS}})
classification_D3_minus_D1 = pd.DataFrame(change_rows)
print('\nD3 minus D1 accuracy changes (percentage points), matched by seed:')
report(classification_D3_minus_D1[['model','seed','accuracy_D3_minus_D1_pp']],digits=6)


FIVE-SEED CLASSIFICATION MEAN AND SAMPLE SD (percentage units):
dataset              model          metric  mean_pct  sample_SD_pct
     D1             REMNET        accuracy 94.125270       0.618480
     D1             REMNET macro_precision 94.221261       0.629384
     D1             REMNET    macro_recall 92.530848       0.879769
     D1             REMNET        macro_f1 93.076180       0.835764
     D1           ResNet50        accuracy 91.619870       1.522640
     D1           ResNet50 macro_precision 91.538130       1.491506
     D1           ResNet50    macro_recall 90.393757       1.821848
     D1           ResNet50        macro_f1 90.599877       1.787162
     D1        MobileNetV2        accuracy 90.583153       0.583553
     D1        MobileNetV2 macro_precision 91.510213       0.630636
     D1        MobileNetV2    macro_recall 88.545438       0.909831
     D1        MobileNetV2        macro_f1 89.298874       0.748192
     D1 EfficientNet-Lite0        accuracy 89.892009

## 3. Paired D3 comparisons: REMNET versus each standalone CNN


In [4]:
difference_rows=[]
result_rows=[]
for metric in METRICS:
    a=np.asarray(classification_values['D3']['REMNET'][metric],dtype=float)
    for baseline in BASELINES:
        b=np.asarray(classification_values['D3'][baseline][metric],dtype=float)
        for seed,rv,bv in zip(SEEDS,a,b):
            difference_rows.append(dict(metric=metric,seed=seed,baseline=baseline,
               REMNET_pct=float(rv),baseline_pct=float(bv),REMNET_minus_baseline_pp=float(rv-bv)))
        result_rows.append(dict(metric=metric,baseline=baseline,
             REMNET_mean_pct=float(a.mean()),REMNET_sample_SD_pct=float(a.std(ddof=1)),
             baseline_mean_pct=float(b.mean()),baseline_sample_SD_pct=float(b.std(ddof=1)),
             **paired_test(a,b)))
classification_seed_differences=pd.DataFrame(difference_rows)
classification_tests=pd.DataFrame(result_rows)
assert len(classification_seed_differences)==60 and len(classification_tests)==12
classification_tests['family']=np.where(classification_tests.metric.isin(['accuracy','macro_f1']),
   'PRIMARY_6_accuracy_macroF1','EXPLORATORY_6_precision_recall')
classification_tests['Holm_p_within_family']=np.nan
for family in classification_tests.family.unique():
    mask=classification_tests.family==family
    classification_tests.loc[mask,'Holm_p_within_family']=holm_adjust(classification_tests.loc[mask,'p_two_sided'].to_numpy())
classification_tests['Holm_p_all_12_sensitivity']=holm_adjust(classification_tests.p_two_sided.to_numpy())
classification_tests['significant_Holm_0_05']=classification_tests.Holm_p_within_family<ALPHA

print('D3 REMNET-MINUS-CNN DIFFERENCES, EACH SEED (percentage points):')
report(classification_seed_differences,digits=6)
print('\nPRIMARY PAIRED INFERENCE (six Holm-adjusted accuracy/macro-F1 tests):')
report(classification_tests[classification_tests.family.str.startswith('PRIMARY')][
 ['metric','baseline','mean_difference','CI95_lower','CI95_upper','Cohen_dz','p_two_sided','Holm_p_within_family','exact_sign_flip_p_two_sided']],digits=9)
print('\nSECONDARY paired inference (macro-precision and macro-recall):')
report(classification_tests[~classification_tests.family.str.startswith('PRIMARY')][
 ['metric','baseline','mean_difference','CI95_lower','CI95_upper','Cohen_dz','p_two_sided','Holm_p_within_family']],digits=9)
print('\nInterpretation: n=5 yields wide uncertainty; exact sign-flip p cannot be smaller than 0.0625 two-sided.')
print('Both datasets / summaries / per-seed differences / statistical tests are in variables displayed above.')


D3 REMNET-MINUS-CNN DIFFERENCES, EACH SEED (percentage points):
         metric  seed           baseline  REMNET_pct  baseline_pct  REMNET_minus_baseline_pp
       accuracy     7           ResNet50   46.096654     34.972834                 11.123820
       accuracy    49           ResNet50   39.576780     40.634830                 -1.058050
       accuracy   112           ResNet50   47.211896     43.580212                  3.631684
       accuracy   345           ResNet50   42.951101     37.861024                  5.090077
       accuracy   476           ResNet50   48.012582     43.751787                  4.260795
       accuracy     7        MobileNetV2   46.096654     37.546468                  8.550186
       accuracy    49        MobileNetV2   39.576780     33.314269                  6.262511
       accuracy   112        MobileNetV2   47.211896     37.689448                  9.522448
       accuracy   345        MobileNetV2   42.951101     41.063769                  1.887332
      